**Q1. Bronze Layer: Raw Ingestion**

In [0]:
catalog = "workspace"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Schemas created successfully.")

Schemas created successfully.


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.telcopay_raw_files")

display(dbutils.fs.ls("/Volumes/workspace/bronze/telcopay_raw_files"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/telcopay_raw_files/customers.csv,customers.csv,293336,1781143359000
dbfs:/Volumes/workspace/bronze/telcopay_raw_files/transactions.csv,transactions.csv,1013559,1781143358000


In [0]:
customers_path = "/Volumes/workspace/bronze/telcopay_raw_files/customers.csv"
transactions_path = "/Volumes/workspace/bronze/telcopay_raw_files/transactions.csv"

bronze_customers_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(customers_path)
)

bronze_transactions_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(transactions_path)
)

bronze_customers_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.bronze.bronze_customers"
)

bronze_transactions_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.bronze.bronze_transactions"
)

print("Bronze ingestion completed from raw CSV files.")
print("bronze_customers row count:", spark.table("workspace.bronze.bronze_customers").count())
print("bronze_transactions row count:", spark.table("workspace.bronze.bronze_transactions").count())

display(spark.table("workspace.bronze.bronze_customers").limit(10))
display(spark.table("workspace.bronze.bronze_transactions").limit(10))x

Bronze ingestion completed from raw CSV files.
bronze_customers row count: 2550
bronze_transactions row count: 12080


customer_id,first_name,last_name,dob,email,phone,plan,state,status,join_date,credit_limit_aud
CUST02433,Priya,Davis,1975-01-11,priya.davis14@gmail.com,61478605776,Postpaid 80,ACT,Active,2019-07-14,1978.82
CUST01930,Raj,Thomas,1964-10-15,raj.thomas89@gmail.com,61443598398,Postpaid 50,VIC,Inactive,2024-06-02,1942.41
CUST00517,Will,Lewis,1971-12-19,will.lewis69@gmail.com,61425304289,Postpaid 80,WA,Suspended,2018-11-30,1633.93
CUST00559,Liam,Allen,1967-11-05,liam.allen67@outlook.com,61480271487,Postpaid 30,ACT,Active,2023-05-29,1426.49
CUST01740,Hassan,Davis,1955-04-29,hassan.davis94@gmail.com,61427640547,Family Pack,WA,Suspended,2024-01-01,1017.96
CUST00555,Chen,Smith,1958-10-03,chen.smith13@gmail.com,61406984775,Business Pro,NT,Inactive,2022-02-27,1459.16
CUST00467,Noah,Hall,1998-08-31,noah.hall39@outlook.com,61495641220,Postpaid 30,NSW,Active,2020-02-10,1760.32
CUST02309,Fatima,Martin,1975-03-17,fatima.martin44@gmail.com,null,Postpaid 80,NSW,Inactive,2020-10-08,1286.17
CUST01641,David,Wilson,1973-09-14,david.wilson19@gmail.com,61416303356,Family Pack,WA,Inactive,2020-04-02,1680.1
CUST00105,Sarah,Lewis,1968-10-06,sarah.lewis87@gmail.com,61455719285,Postpaid 80,TAS,Suspended,2021-02-15,724.34


transaction_id,customer_id,txn_type,amount_aud,duration_sec,data_mb,channel,txn_date,status,description
TXN0007549,CUST01940,Data,14.21,null,3090.2,App,2023-02-25,Failed,Data charge
TXN0004009,CUST00825,Recharge,30.0,null,null,App,2024-01-04,Completed,Recharge charge
TXN0011571,CUST01985,Data,39.43,null,2667.0,IVR,2023-05-15,Completed,Data charge
TXN0001878,CUST02439,Roaming Data,42.99,null,4605.2,Agent,2024-04-16,Completed,Roaming Data charge
TXN0010314,CUST00835,Recharge,30.0,null,null,Store,2023-02-14,Pending,Recharge charge
TXN0010299,CUST01448,Call,18.01,2166,null,IVR,2024-01-28,Reversed,Call charge
TXN0009703,CUST01738,Roaming Data,38.03,null,159.6,Agent,2023-10-21,Reversed,Roaming Data charge
TXN0006325,CUST02115,Call,17.95,1447,null,Agent,2024-04-20,Completed,Call charge
TXN0003861,CUST02491,Data,67.78,null,3591.3,Agent,2023-10-20,Pending,Data charge
TXN0001581,CUST02172,Data,23.89,null,2067.3,Agent,2023-04-06,Pending,Data charge


In [0]:
bronze_customers = spark.table("workspace.bronze.bronze_customers")
bronze_transactions = spark.table("workspace.bronze.bronze_transactions")

print("bronze_customers row count:", bronze_customers.count())
print("bronze_transactions row count:", bronze_transactions.count())

display(bronze_customers.limit(10))
display(bronze_transactions.limit(10))

bronze_customers row count: 2550
bronze_transactions row count: 12080


customer_id,first_name,last_name,dob,email,phone,plan,state,status,join_date,credit_limit_aud
CUST02433,Priya,Davis,1975-01-11,priya.davis14@gmail.com,61478605776,Postpaid 80,ACT,Active,2019-07-14,1978.82
CUST01930,Raj,Thomas,1964-10-15,raj.thomas89@gmail.com,61443598398,Postpaid 50,VIC,Inactive,2024-06-02,1942.41
CUST00517,Will,Lewis,1971-12-19,will.lewis69@gmail.com,61425304289,Postpaid 80,WA,Suspended,2018-11-30,1633.93
CUST00559,Liam,Allen,1967-11-05,liam.allen67@outlook.com,61480271487,Postpaid 30,ACT,Active,2023-05-29,1426.49
CUST01740,Hassan,Davis,1955-04-29,hassan.davis94@gmail.com,61427640547,Family Pack,WA,Suspended,2024-01-01,1017.96
CUST00555,Chen,Smith,1958-10-03,chen.smith13@gmail.com,61406984775,Business Pro,NT,Inactive,2022-02-27,1459.16
CUST00467,Noah,Hall,1998-08-31,noah.hall39@outlook.com,61495641220,Postpaid 30,NSW,Active,2020-02-10,1760.32
CUST02309,Fatima,Martin,1975-03-17,fatima.martin44@gmail.com,null,Postpaid 80,NSW,Inactive,2020-10-08,1286.17
CUST01641,David,Wilson,1973-09-14,david.wilson19@gmail.com,61416303356,Family Pack,WA,Inactive,2020-04-02,1680.1
CUST00105,Sarah,Lewis,1968-10-06,sarah.lewis87@gmail.com,61455719285,Postpaid 80,TAS,Suspended,2021-02-15,724.34


transaction_id,customer_id,txn_type,amount_aud,duration_sec,data_mb,channel,txn_date,status,description
TXN0007549,CUST01940,Data,14.21,null,3090.2,App,2023-02-25,Failed,Data charge
TXN0004009,CUST00825,Recharge,30.0,null,null,App,2024-01-04,Completed,Recharge charge
TXN0011571,CUST01985,Data,39.43,null,2667.0,IVR,2023-05-15,Completed,Data charge
TXN0001878,CUST02439,Roaming Data,42.99,null,4605.2,Agent,2024-04-16,Completed,Roaming Data charge
TXN0010314,CUST00835,Recharge,30.0,null,null,Store,2023-02-14,Pending,Recharge charge
TXN0010299,CUST01448,Call,18.01,2166,null,IVR,2024-01-28,Reversed,Call charge
TXN0009703,CUST01738,Roaming Data,38.03,null,159.6,Agent,2023-10-21,Reversed,Roaming Data charge
TXN0006325,CUST02115,Call,17.95,1447,null,Agent,2024-04-20,Completed,Call charge
TXN0003861,CUST02491,Data,67.78,null,3591.3,Agent,2023-10-20,Pending,Data charge
TXN0001581,CUST02172,Data,23.89,null,2067.3,Agent,2023-04-06,Pending,Data charge


In [0]:
from pyspark.sql.functions import col, count, when, lower, trim

# 1. Duplicate primary/business keys
duplicate_customers = bronze_customers.groupBy("customer_id").count().filter(col("count") > 1)
duplicate_transactions = bronze_transactions.groupBy("transaction_id").count().filter(col("count") > 1)

print("Duplicate customer_id count:", duplicate_customers.count())
print("Duplicate transaction_id count:", duplicate_transactions.count())

display(duplicate_customers.orderBy(col("count").desc()).limit(10))
display(duplicate_transactions.orderBy(col("count").desc()).limit(10))

# 2. Missing values by column
customers_nulls = bronze_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in bronze_customers.columns
])

transactions_nulls = bronze_transactions.select([
    count(when(col(c).isNull(), c)).alias(c) for c in bronze_transactions.columns
])

print("Missing values in bronze_customers")
display(customers_nulls)

print("Missing values in bronze_transactions")
display(transactions_nulls)

# 3. Inconsistent status values
print("Customer status variants")
display(
    bronze_customers
    .groupBy("status")
    .count()
    .orderBy("status")
)

print("Transaction status variants")
display(
    bronze_transactions
    .groupBy("status")
    .count()
    .orderBy("status")
)

# 4. Negative numeric values
print("Negative credit limits")
display(
    bronze_customers
    .filter(col("credit_limit_aud") < 0)
    .select("customer_id", "credit_limit_aud")
    .limit(20)
)

print("Negative transaction amounts")
display(
    bronze_transactions
    .filter(col("amount_aud") < 0)
    .select("transaction_id", "customer_id", "amount_aud", "txn_type", "status")
    .limit(20)
)

# 5. Non-standard date formats
print("Non-standard join_date format")
display(
    bronze_customers
    .filter(~col("join_date").cast("string").rlike("^\\d{4}-\\d{2}-\\d{2}$"))
    .select("customer_id", "join_date")
    .limit(20)
)

print("Non-standard txn_date format")
display(
    bronze_transactions
    .filter(~col("txn_date").cast("string").rlike("^\\d{4}-\\d{2}-\\d{2}$"))
    .select("transaction_id", "txn_date")
    .limit(20)
)

Duplicate customer_id count: 48
Duplicate transaction_id count: 80


customer_id,count
CUST00303,3
CUST01406,3
CUST01249,2
CUST01989,2
CUST00056,2
CUST00325,2
CUST00114,2
CUST00599,2
CUST01231,2
CUST01903,2


transaction_id,count
TXN0005900,2
TXN0009821,2
TXN0006209,2
TXN0005248,2
TXN0004601,2
TXN0007587,2
TXN0001367,2
TXN0006242,2
TXN0007070,2
TXN0010757,2


Missing values in bronze_customers


customer_id,first_name,last_name,dob,email,phone,plan,state,status,join_date,credit_limit_aud
0,31,0,0,80,73,0,30,0,0,0


...

transaction_id,customer_id,amount_aud,txn_type,status
TXN0003327,CUST00292,-37.89,SMS,Completed
TXN0006692,CUST02142,-142.91,Roaming Data,Completed
TXN0008208,CUST00510,-79.07,International Call,Completed
TXN0000788,CUST00406,-44.23,Data,Completed
TXN0003732,CUST02159,-68.48,Recharge,Completed
TXN0002459,CUST01209,-80.66,SMS,Completed
TXN0011228,CUST02207,-17.91,International Call,Reversed
TXN0000012,CUST01086,-100.04,Roaming Data,Completed
TXN0000549,CUST01505,-133.57,Data,Completed
TXN0002664,CUST01497,-42.27,Roaming Data,Reversed


Non-standard join_date format


customer_id,join_date
CUST00723,28/01/2020
CUST02255,13/05/2018
CUST01275,17/08/2020
CUST02316,08/03/2020
CUST01510,06/07/2019
CUST01220,25/05/2021
CUST01893,30/05/2019
CUST02061,11/02/2020
CUST02280,27/02/2019
CUST00728,08/04/2020


Non-standard txn_date format


transaction_id,txn_date
TXN0009766,04-07-2023
TXN0001750,15-02-2024
TXN0006361,20-02-2023
TXN0007587,07-11-2024
TXN0008178,01-07-2023
TXN0009912,14-03-2023
TXN0003712,24-01-2023
TXN0007333,05-02-2023
TXN0008571,17-11-2023
TXN0009348,04-05-2023


**Q2 Silver Layer: Cleansing and Standardisation**

In [0]:
from pyspark.sql import functions as F

silver_customers = spark.table("workspace.silver.silver_customers")
silver_transactions = spark.table("workspace.silver.silver_transactions")

invalid_customer_id_transactions = (
    silver_transactions.alias("t")
    .filter(F.col("t.customer_id").isNotNull())
    .join(
        silver_customers.select("customer_id").alias("c"),
        F.col("t.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
)

print("Transactions with customer_id not found in silver_customers:", invalid_customer_id_transactions.count())

display(
    invalid_customer_id_transactions
    .select("transaction_id", "customer_id", "txn_type", "amount_aud", "channel", "txn_date", "status")
    .limit(20)
)

Transactions with customer_id not found in silver_customers: 0


transaction_id,customer_id,txn_type,amount_aud,channel,txn_date,status


In [0]:
from pyspark.sql.functions import (
    col, trim, lower, upper, initcap, regexp_replace, when, lit,
    coalesce, desc, expr
)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Load Bronze tables
bronze_customers = spark.table("workspace.bronze.bronze_customers")
bronze_transactions = spark.table("workspace.bronze.bronze_transactions")

print("Before cleansing")
print("bronze_customers row count:", bronze_customers.count())
print("bronze_transactions row count:", bronze_transactions.count())

# Helper function: safely parse mixed date formats
def parse_mixed_date(column_name):
    return coalesce(
        expr(f"try_to_date(cast({column_name} as string), 'yyyy-MM-dd')"),
        expr(f"try_to_date(cast({column_name} as string), 'dd/MM/yyyy')"),
        expr(f"try_to_date(cast({column_name} as string), 'dd-MM-yyyy')")
    )

# -----------------------------
# Silver Customers
# -----------------------------

customers_standardised = (
    bronze_customers
    .withColumn("customer_id", upper(trim(col("customer_id"))))
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn(
        "phone",
        regexp_replace(
            regexp_replace(trim(col("phone").cast("string")), "\\.0$", ""),
            "[^0-9]",
            ""
        )
    )
    .withColumn("plan", trim(col("plan")))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("status", initcap(lower(trim(col("status")))))
    .withColumn("dob_raw", col("dob").cast("string"))
    .withColumn("join_date_raw", col("join_date").cast("string"))
    .withColumn("dob", parse_mixed_date("dob_raw"))
    .withColumn("join_date", parse_mixed_date("join_date_raw"))
    .withColumn("credit_limit_aud_raw", col("credit_limit_aud"))
    .withColumn("credit_limit_aud", col("credit_limit_aud").cast("double"))
    .withColumn(
        "invalid_credit_limit_flag",
        when(col("credit_limit_aud") < 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "credit_limit_aud",
        when(col("credit_limit_aud") < 0, lit(None).cast("double"))
        .otherwise(col("credit_limit_aud"))
    )
    .withColumn(
        "valid_state_flag",
        when(col("state").isin("ACT", "NSW", "NT", "QLD", "SA", "TAS", "VIC", "WA"), lit(True))
        .otherwise(lit(False))
    )
)

customers_for_dedup = customers_standardised.withColumn(
    "customer_completeness_score",
    (
        when(col("first_name").isNotNull(), 1).otherwise(0) +
        when(col("last_name").isNotNull(), 1).otherwise(0) +
        when(col("email").isNotNull(), 1).otherwise(0) +
        when(col("phone").isNotNull(), 1).otherwise(0) +
        when(col("state").isNotNull(), 1).otherwise(0) +
        when(col("status").isNotNull(), 1).otherwise(0) +
        when(col("join_date").isNotNull(), 1).otherwise(0) +
        when(col("credit_limit_aud").isNotNull(), 1).otherwise(0)
    )
)

customer_window = Window.partitionBy("customer_id").orderBy(
    desc("customer_completeness_score"),
    desc("join_date")
)

silver_customers = (
    customers_for_dedup
    .withColumn("row_number", row_number().over(customer_window))
    .filter(col("row_number") == 1)
    .drop("row_number", "customer_completeness_score")
)

# -----------------------------
# Silver Transactions
# -----------------------------

transactions_standardised = (
    bronze_transactions
    .withColumn("transaction_id", upper(trim(col("transaction_id"))))
    .withColumn("customer_id", upper(trim(col("customer_id"))))
    .withColumn("txn_type", initcap(trim(col("txn_type"))))
    .withColumn("channel", initcap(trim(col("channel"))))
    .withColumn("status", initcap(lower(trim(col("status")))))
    .withColumn("description", trim(col("description")))
    .withColumn("txn_date_raw", col("txn_date").cast("string"))
    .withColumn("txn_date", parse_mixed_date("txn_date_raw"))
    .withColumn("amount_aud_raw", col("amount_aud"))
    .withColumn("amount_aud", col("amount_aud").cast("double"))
    .withColumn("duration_sec", col("duration_sec").cast("long"))
    .withColumn("data_mb", col("data_mb").cast("double"))
    .withColumn(
        "negative_amount_flag",
        when(col("amount_aud") < 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "valid_transaction_status_flag",
        when(col("status").isin("Completed", "Failed", "Pending", "Reversed"), lit(True))
        .otherwise(lit(False))
    )
    .withColumn(
        "valid_channel_flag",
        when(col("channel").isin("App", "Store", "Ivr", "Agent"), lit(True))
        .otherwise(lit(False))
    )
)

transactions_for_dedup = transactions_standardised.withColumn(
    "transaction_completeness_score",
    (
        when(col("customer_id").isNotNull(), 1).otherwise(0) +
        when(col("txn_type").isNotNull(), 1).otherwise(0) +
        when(col("amount_aud").isNotNull(), 1).otherwise(0) +
        when(col("channel").isNotNull(), 1).otherwise(0) +
        when(col("txn_date").isNotNull(), 1).otherwise(0) +
        when(col("status").isNotNull(), 1).otherwise(0)
    )
)

transaction_window = Window.partitionBy("transaction_id").orderBy(
    desc("transaction_completeness_score"),
    desc("txn_date")
)

silver_transactions = (
    transactions_for_dedup
    .withColumn("row_number", row_number().over(transaction_window))
    .filter(col("row_number") == 1)
    .drop("row_number", "transaction_completeness_score")
)

# Write Silver Delta tables
silver_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.silver_customers")
silver_transactions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.silver_transactions")

print("Silver tables created successfully.")
print("silver_customers row count:", silver_customers.count())
print("silver_transactions row count:", silver_transactions.count())

Before cleansing
bronze_customers row count: 2550
bronze_transactions row count: 12080
Silver tables created successfully.
silver_customers row count: 2500
silver_transactions row count: 12000


In [0]:
from pyspark.sql import functions as F

bronze_customers = spark.table("workspace.bronze.bronze_customers")
bronze_transactions = spark.table("workspace.bronze.bronze_transactions")
silver_customers = spark.table("workspace.silver.silver_customers")
silver_transactions = spark.table("workspace.silver.silver_transactions")

def duplicate_key_count(df, key_col):
    return df.groupBy(key_col).count().filter(F.col("count") > 1).count()

def status_summary(df, status_col):
    rows = (
        df.groupBy(status_col)
        .count()
        .orderBy(status_col)
        .collect()
    )
    return ", ".join([f"{r[status_col]}={r['count']}" for r in rows])

invalid_customer_id_transactions = (
    silver_transactions.alias("t")
    .filter(F.col("t.customer_id").isNotNull() & (F.col("t.customer_id") != ""))
    .join(
        silver_customers.select("customer_id").alias("c"),
        F.col("t.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
)

missing_customer_id_transactions = (
    silver_transactions
    .filter(F.col("customer_id").isNull() | (F.col("customer_id") == ""))
)

summary_rows = [
    ("Customers", "Bronze row count", str(bronze_customers.count()), "Raw customer rows before cleansing"),
    ("Customers", "Silver row count", str(silver_customers.count()), "Cleaned customer rows after deduplication and standardisation"),
    ("Customers", "Rows removed", str(bronze_customers.count() - silver_customers.count()), "Removed mainly due to duplicate customer IDs"),
    ("Customers", "Duplicate customer_id after Silver", str(duplicate_key_count(silver_customers, "customer_id")), "Expected result is 0"),
    ("Customers", "Customer status after standardisation", status_summary(silver_customers, "status"), "Status values are normalised"),
    ("Customers", "Invalid credit limits flagged", str(silver_customers.filter(F.col("invalid_credit_limit_flag") == True).count()), "Negative credit limits retained as flagged anomalies"),
    ("Customers", "Unparsed join dates after Silver", str(silver_customers.filter(F.col("join_date").isNull() & F.col("join_date_raw").isNotNull()).count()), "Expected result is 0"),

    ("Transactions", "Bronze row count", str(bronze_transactions.count()), "Raw transaction rows before cleansing"),
    ("Transactions", "Silver row count", str(silver_transactions.count()), "Cleaned transaction rows after deduplication and standardisation"),
    ("Transactions", "Rows removed", str(bronze_transactions.count() - silver_transactions.count()), "Removed mainly due to duplicate transaction IDs"),
    ("Transactions", "Duplicate transaction_id after Silver", str(duplicate_key_count(silver_transactions, "transaction_id")), "Expected result is 0"),
    ("Transactions", "Transaction status after standardisation", status_summary(silver_transactions, "status"), "Status values are normalised"),
    ("Transactions", "Missing customer_id retained", str(missing_customer_id_transactions.count()), "These cannot be assigned to a customer-level summary"),
    ("Transactions", "Non-null customer_id not found in silver_customers", str(invalid_customer_id_transactions.count()), "Referential integrity check after cleaning"),
    ("Transactions", "Negative amounts flagged", str(silver_transactions.filter(F.col("negative_amount_flag") == True).count()), "Negative transaction amounts retained as flagged anomalies"),
    ("Transactions", "Unparsed transaction dates after Silver", str(silver_transactions.filter(F.col("txn_date").isNull() & F.col("txn_date_raw").isNotNull()).count()), "Expected result is 0")
]

silver_quality_summary = spark.createDataFrame(
    summary_rows,
    ["Table", "Validation Check", "Result", "Interpretation"]
)

display(silver_quality_summary)

Table,Validation Check,Result,Interpretation
Customers,Bronze row count,2550,Raw customer rows before cleansing
Customers,Silver row count,2500,Cleaned customer rows after deduplication and standardisation
Customers,Rows removed,50,Removed mainly due to duplicate customer IDs
Customers,Duplicate customer_id after Silver,0,Expected result is 0
Customers,Customer status after standardisation,"Active=818, Inactive=815, Suspended=867",Status values are normalised
Customers,Invalid credit limits flagged,30,Negative credit limits retained as flagged anomalies
Customers,Unparsed join dates after Silver,0,Expected result is 0
Transactions,Bronze row count,12080,Raw transaction rows before cleansing
Transactions,Silver row count,12000,Cleaned transaction rows after deduplication and standardisation
Transactions,Rows removed,80,Removed mainly due to duplicate transaction IDs


In [0]:
from pyspark.sql.functions import col, count, when

silver_customers = spark.table("workspace.silver.silver_customers")
silver_transactions = spark.table("workspace.silver.silver_transactions")

# 1. Confirm no duplicate IDs remain
duplicate_silver_customers = (
    silver_customers
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_silver_transactions = (
    silver_transactions
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate customer_id in silver_customers:", duplicate_silver_customers.count())
print("Duplicate transaction_id in silver_transactions:", duplicate_silver_transactions.count())

# 2. Check standardised status values
print("Customer status after Silver standardisation")
display(
    silver_customers
    .groupBy("status")
    .count()
    .orderBy("status")
)

print("Transaction status after Silver standardisation")
display(
    silver_transactions
    .groupBy("status")
    .count()
    .orderBy("status")
)

# 3. Check date parsing result
print("Unparsed customer join dates")
display(
    silver_customers
    .filter(col("join_date").isNull() & col("join_date_raw").isNotNull())
    .select("customer_id", "join_date_raw", "join_date")
    .limit(20)
)

print("Unparsed transaction dates")
display(
    silver_transactions
    .filter(col("txn_date").isNull() & col("txn_date_raw").isNotNull())
    .select("transaction_id", "txn_date_raw", "txn_date")
    .limit(20)
)

# 4. Show flagged financial anomalies
print("Invalid credit limits flagged in silver_customers")
display(
    silver_customers
    .filter(col("invalid_credit_limit_flag") == True)
    .select("customer_id", "credit_limit_aud_raw", "credit_limit_aud", "invalid_credit_limit_flag")
    .limit(20)
)

print("Negative transaction amounts flagged in silver_transactions")
display(
    silver_transactions
    .filter(col("negative_amount_flag") == True)
    .select("transaction_id", "customer_id", "amount_aud_raw", "amount_aud", "status", "negative_amount_flag")
    .limit(20)
)

Duplicate customer_id in silver_customers: 0
Duplicate transaction_id in silver_transactions: 0
Customer status after Silver standardisation


status,count
Active,818
Inactive,815
Suspended,867


Transaction status after Silver standardisation


status,count
Completed,8352
Failed,1261
Pending,1463
Reversed,924


Unparsed customer join dates


...

transaction_id,txn_date_raw,txn_date


Invalid credit limits flagged in silver_customers


customer_id,credit_limit_aud_raw,credit_limit_aud,invalid_credit_limit_flag
CUST00143,-325.01,null,true
CUST00197,-334.28,null,true
CUST00382,-203.27,null,true
CUST00409,-181.01,null,true
CUST00437,-65.63,null,true
CUST00525,-429.34,null,true
CUST00676,-254.1,null,true
CUST00744,-93.4,null,true
CUST00765,-406.49,null,true
CUST00777,-339.28,null,true


Negative transaction amounts flagged in silver_transactions


transaction_id,customer_id,amount_aud_raw,amount_aud,status,negative_amount_flag
TXN0000012,CUST01086,-100.04,-100.04,Completed,true
TXN0000466,CUST01682,-49.51,-49.51,Completed,true
TXN0000520,CUST00360,-106.21,-106.21,Pending,true
TXN0000549,CUST01505,-133.57,-133.57,Completed,true
TXN0000678,CUST01189,-54.88,-54.88,Completed,true
TXN0000788,CUST00406,-44.23,-44.23,Completed,true
TXN0000839,CUST00739,-118.3,-118.3,Completed,true
TXN0000957,CUST00891,-67.31,-67.31,Failed,true
TXN0000964,CUST00915,-62.52,-62.52,Completed,true
TXN0001282,CUST01268,-115.04,-115.04,Completed,true


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_customers = spark.table("workspace.silver.silver_customers")
silver_transactions = spark.table("workspace.silver.silver_transactions")

# Prepare clean customer and transaction views to avoid ambiguous status columns
customers = (
    silver_customers
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "plan",
        "state",
        F.col("status").alias("customer_status"),
        "join_date",
        "credit_limit_aud"
    )
)

transactions = (
    silver_transactions
    .select(
        "transaction_id",
        "customer_id",
        "txn_type",
        "amount_aud",
        "duration_sec",
        "data_mb",
        "channel",
        "txn_date",
        F.col("status").alias("transaction_status"),
        "negative_amount_flag"
    )
)

# -------------------------------------------------------------------
# Gold Table 1: Customer-level value and risk segmentation
# -------------------------------------------------------------------

customer_txn_joined = (
    customers
    .join(transactions, on="customer_id", how="left")
)

gold_customer_value_segments_base = (
    customer_txn_joined
    .groupBy(
        "customer_id",
        "first_name",
        "last_name",
        "plan",
        "state",
        "customer_status",
        "join_date",
        "credit_limit_aud"
    )
    .agg(
        F.countDistinct("transaction_id").alias("total_transaction_count"),

        F.sum(F.when(F.col("transaction_status") == "Completed", 1).otherwise(0))
            .alias("completed_transaction_count"),

        F.sum(F.when(F.col("transaction_status") == "Failed", 1).otherwise(0))
            .alias("failed_transaction_count"),

        F.sum(F.when(F.col("transaction_status") == "Pending", 1).otherwise(0))
            .alias("pending_transaction_count"),

        F.sum(F.when(F.col("transaction_status") == "Reversed", 1).otherwise(0))
            .alias("reversed_transaction_count"),

        F.round(
            F.sum(
                F.when(
                    (F.col("transaction_status") == "Completed") &
                    (F.col("amount_aud") > 0),
                    F.col("amount_aud")
                ).otherwise(0)
            ),
            2
        ).alias("total_completed_spend_aud"),

        F.round(
            F.avg(
                F.when(
                    (F.col("transaction_status") == "Completed") &
                    (F.col("amount_aud") > 0),
                    F.col("amount_aud")
                )
            ),
            2
        ).alias("avg_completed_transaction_aud"),

        F.sum(F.when(F.col("negative_amount_flag") == True, 1).otherwise(0))
            .alias("negative_amount_count"),

        F.round(F.sum(F.when(F.col("data_mb") > 0, F.col("data_mb")).otherwise(0)), 2)
            .alias("total_data_mb"),

        F.sum(F.when(F.col("duration_sec") > 0, F.col("duration_sec")).otherwise(0))
            .alias("total_duration_sec"),

        F.min("txn_date").alias("first_txn_date"),
        F.max("txn_date").alias("last_txn_date")
    )
)

# Create value segments using ranking based on completed spend
value_window = Window.orderBy(F.desc("total_completed_spend_aud"))

gold_customer_value_segments = (
    gold_customer_value_segments_base
    .withColumn("value_rank_bucket", F.ntile(3).over(value_window))
    .withColumn(
        "customer_value_segment",
        F.when(F.col("value_rank_bucket") == 1, F.lit("High Value"))
         .when(F.col("value_rank_bucket") == 2, F.lit("Medium Value"))
         .otherwise(F.lit("Low Value"))
    )
    .withColumn(
        "risk_segment",
        F.when(
            (F.col("customer_status") != "Active") &
            (F.col("total_completed_spend_aud") > 0),
            F.lit("Retention Attention")
        )
        .when(
            (F.col("failed_transaction_count") + F.col("pending_transaction_count")) > 0,
            F.lit("Transaction Attention")
        )
        .otherwise(F.lit("Normal"))
    )
    .drop("value_rank_bucket")
)

gold_customer_value_segments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.gold_customer_value_segments"
)

# -------------------------------------------------------------------
# Gold Table 2: Monthly segment performance
# -------------------------------------------------------------------

transaction_segment_joined = (
    transactions
    .join(
        customers.select("customer_id", "plan", "state", "customer_status"),
        on="customer_id",
        how="left"
    )
    .withColumn("txn_month", F.date_trunc("month", F.col("txn_date")).cast("date"))
)

gold_monthly_segment_performance = (
    transaction_segment_joined
    .groupBy("txn_month", "channel", "plan", "state")
    .agg(
        F.countDistinct("transaction_id").alias("total_transactions"),

        F.sum(F.when(F.col("transaction_status") == "Completed", 1).otherwise(0))
            .alias("completed_transactions"),

        F.sum(F.when(F.col("transaction_status") == "Failed", 1).otherwise(0))
            .alias("failed_transactions"),

        F.sum(F.when(F.col("transaction_status") == "Pending", 1).otherwise(0))
            .alias("pending_transactions"),

        F.sum(F.when(F.col("transaction_status") == "Reversed", 1).otherwise(0))
            .alias("reversed_transactions"),

        F.countDistinct("customer_id").alias("active_customers"),

        F.round(
            F.sum(
                F.when(
                    (F.col("transaction_status") == "Completed") &
                    (F.col("amount_aud") > 0),
                    F.col("amount_aud")
                ).otherwise(0)
            ),
            2
        ).alias("completed_revenue_aud"),

        F.round(
            F.sum(
                F.when(
                    (F.col("transaction_status").isin("Failed", "Pending", "Reversed")) &
                    (F.col("amount_aud") > 0),
                    F.col("amount_aud")
                ).otherwise(0)
            ),
            2
        ).alias("revenue_at_risk_aud"),

        F.sum(F.when(F.col("negative_amount_flag") == True, 1).otherwise(0))
            .alias("negative_amount_count")
    )
    .withColumn(
        "failure_rate_pct",
        F.round((F.col("failed_transactions") / F.col("total_transactions")) * 100, 2)
    )
    .withColumn(
        "pending_rate_pct",
        F.round((F.col("pending_transactions") / F.col("total_transactions")) * 100, 2)
    )
    .withColumn(
        "reversal_rate_pct",
        F.round((F.col("reversed_transactions") / F.col("total_transactions")) * 100, 2)
    )
)

gold_monthly_segment_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.gold.gold_monthly_segment_performance"
)

print("Gold tables created successfully.")
print("gold_customer_value_segments row count:", spark.table("workspace.gold.gold_customer_value_segments").count())
print("gold_monthly_segment_performance row count:", spark.table("workspace.gold.gold_monthly_segment_performance").count())

display(
    spark.table("workspace.gold.gold_customer_value_segments")
    .orderBy(F.desc("total_completed_spend_aud"))
    .limit(20)
)

display(
    spark.table("workspace.gold.gold_monthly_segment_performance")
    .orderBy("txn_month", "channel", "plan", "state")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold tables created successfully.
gold_customer_value_segments row count: 2500
gold_monthly_segment_performance row count: 5240


customer_id,first_name,last_name,plan,state,customer_status,join_date,credit_limit_aud,total_transaction_count,completed_transaction_count,failed_transaction_count,pending_transaction_count,reversed_transaction_count,total_completed_spend_aud,avg_completed_transaction_aud,negative_amount_count,total_data_mb,total_duration_sec,first_txn_date,last_txn_date,customer_value_segment,risk_segment
CUST00649,Fatima,Walker,Postpaid 50,NT,Active,2019-04-15,605.84,7,6,1,0,0,2707.16,451.19,0,8018.0,11530,2023-04-06,2024-12-13,High Value,Transaction Attention
CUST02412,Oliver,Allen,Family Pack,WA,Active,2022-02-21,984.77,5,3,1,1,0,2537.92,845.97,0,4307.4,2744,2023-06-22,2024-10-11,High Value,Transaction Attention
CUST01517,Aisha,Walker,Prepaid Basic,VIC,Suspended,2022-10-21,837.81,11,9,1,1,0,2496.79,277.42,0,20612.8,1290,2023-03-19,2025-12-11,High Value,Retention Attention
CUST02152,null,Miller,Family Pack,WA,Suspended,2024-04-14,1381.68,4,4,0,0,0,2451.9,612.97,0,0.0,4917,2023-06-10,2024-10-18,High Value,Retention Attention
CUST02045,Tom,Walker,Postpaid 30,ACT,Suspended,2019-10-07,1039.63,7,3,2,2,0,2381.09,793.7,0,8270.4,0,2023-06-18,2024-12-20,High Value,Retention Attention
CUST01742,Mohammed,Wilson,Postpaid 30,TAS,Suspended,2020-12-11,779.44,1,1,0,0,0,2198.98,2198.98,0,0.0,3027,2024-10-02,2024-10-02,High Value,Retention Attention
CUST01159,Emma,Jones,Business Pro,TAS,Suspended,2021-04-17,933.84,3,2,1,0,0,1979.87,989.94,0,0.0,1518,2024-02-14,2024-12-08,High Value,Retention Attention
CUST02352,Grace,Smith,Postpaid 80,VIC,Active,2019-09-17,1431.32,5,4,1,0,0,1846.98,461.75,0,7060.5,2308,2023-06-02,2024-12-27,High Value,Transaction Attention
CUST00140,Zoe,Jones,Business Pro,NT,Inactive,2024-04-01,1200.99,9,6,1,0,2,1826.34,304.39,0,13634.5,9546,2023-03-07,2024-11-14,High Value,Retention Attention
CUST00846,Will,Brown,Business Pro,VIC,Active,2021-08-31,954.08,3,3,0,0,0,1658.89,552.96,0,2205.0,2270,2023-01-03,2023-08-04,High Value,Normal


txn_month,channel,plan,state,total_transactions,completed_transactions,failed_transactions,pending_transactions,reversed_transactions,active_customers,completed_revenue_aud,revenue_at_risk_aud,negative_amount_count,failure_rate_pct,pending_rate_pct,reversal_rate_pct
2023-01-01,null,Business Pro,QLD,1,1,0,0,0,1,55.79,0.0,0,0.0,0.0,0.0
2023-01-01,null,Postpaid 50,VIC,1,0,0,0,1,1,0.0,17.1,0,0.0,0.0,100.0
2023-01-01,Agent,null,null,2,1,0,0,1,0,77.58,50.0,0,0.0,0.0,50.0
2023-01-01,Agent,Business Pro,ACT,1,1,0,0,0,1,18.03,0.0,0,0.0,0.0,0.0
2023-01-01,Agent,Business Pro,NSW,2,1,1,0,0,2,69.16,23.05,0,50.0,0.0,0.0
2023-01-01,Agent,Business Pro,NT,3,2,0,1,0,2,29.01,103.95,0,0.0,33.33,0.0
2023-01-01,Agent,Business Pro,QLD,2,0,0,1,1,2,0.0,22.46,0,0.0,50.0,50.0
2023-01-01,Agent,Business Pro,SA,2,2,0,0,0,2,174.03,0.0,0,0.0,0.0,0.0
2023-01-01,Agent,Business Pro,TAS,3,2,0,0,1,3,50.0,81.76,0,0.0,0.0,33.33
2023-01-01,Agent,Business Pro,VIC,5,2,1,1,1,5,69.69,96.85,0,20.0,20.0,20.0


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql import functions as F

gold_monthly_segment_performance = spark.table("workspace.gold.gold_monthly_segment_performance")
gold_customer_value_segments = spark.table("workspace.gold.gold_customer_value_segments")

# Chart 1: CEO / CFO - Revenue at risk by channel
chart_revenue_at_risk_by_channel = (
    gold_monthly_segment_performance
    .groupBy("channel")
    .agg(
        F.round(F.sum("revenue_at_risk_aud"), 2).alias("revenue_at_risk_aud"),
        F.round(F.sum("completed_revenue_aud"), 2).alias("completed_revenue_aud"),
        F.sum("total_transactions").alias("total_transactions")
    )
    .orderBy(F.desc("revenue_at_risk_aud"))
)

print("Chart 1: Revenue at risk by channel")
display(chart_revenue_at_risk_by_channel)

# Chart 2: COO / CEO - Failure rate by channel
chart_failure_rate_by_channel = (
    gold_monthly_segment_performance
    .groupBy("channel")
    .agg(
        F.sum("total_transactions").alias("total_transactions"),
        F.sum("failed_transactions").alias("failed_transactions")
    )
    .withColumn(
        "failure_rate_pct",
        F.round((F.col("failed_transactions") / F.col("total_transactions")) * 100, 2)
    )
    .orderBy(F.desc("failure_rate_pct"))
)

print("Chart 2: Failure rate by channel")
display(chart_failure_rate_by_channel)

# Chart 3: CEO - Completed revenue trend by month and channel
# Chart 3: CEO - Completed revenue trend by top 3 channels

top_3_channels = (
    gold_monthly_segment_performance
    .filter(F.col("channel").isNotNull())
    .groupBy("channel")
    .agg(
        F.sum("completed_revenue_aud").alias("total_completed_revenue_aud")
    )
    .orderBy(F.desc("total_completed_revenue_aud"))
    .limit(3)
    .select("channel")
)

chart_completed_revenue_top_3_channels = (
    gold_monthly_segment_performance
    .filter(F.col("channel").isNotNull())
    .join(top_3_channels, on="channel", how="inner")
    .groupBy("txn_month", "channel")
    .agg(
        F.round(F.sum("completed_revenue_aud"), 2).alias("completed_revenue_aud")
    )
    .orderBy("txn_month", "channel")
)

print("Chart 3: Completed revenue trend by top 3 channels")
display(chart_completed_revenue_top_3_channels)

# Optional supporting table: customer value segment distribution
chart_customer_value_by_plan = (
    gold_customer_value_segments
    .groupBy("plan", "customer_value_segment")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.round(F.sum("total_completed_spend_aud"), 2).alias("completed_revenue_aud")
    )
    .orderBy("plan", "customer_value_segment")
)

print("Optional: Customer value segment by plan")
display(chart_customer_value_by_plan)

Chart 1: Revenue at risk by channel


channel,revenue_at_risk_aud,completed_revenue_aud,total_transactions
Ivr,28865.68,65416.37,2420
Website,27910.33,64339.17,2458
App,27701.54,63536.38,2330
Agent,27118.98,63090.84,2430
Store,24626.81,62066.6,2322
null,342.32,914.27,40


Databricks visualization. Run in Databricks to view.

Chart 2: Failure rate by channel


channel,total_transactions,failed_transactions,failure_rate_pct
null,40,6,15.0
Agent,2430,292,12.02
Ivr,2420,263,10.87
App,2330,240,10.3
Store,2322,228,9.82
Website,2458,232,9.44


...

Chart 3: Completed revenue trend by top 3 channels


txn_month,channel,completed_revenue_aud
2023-01-01,App,2595.61
2023-01-01,Ivr,2387.57
2023-01-01,Website,2646.58
2023-02-01,App,2203.16
2023-02-01,Ivr,1163.95
2023-02-01,Website,2439.91
2023-03-01,App,2675.56
2023-03-01,Ivr,4803.13
2023-03-01,Website,3022.2
2023-04-01,App,2580.36


Databricks visualization. Run in Databricks to view.

Optional: Customer value segment by plan


plan,customer_value_segment,customer_count,completed_revenue_aud
Business Pro,High Value,149,41147.34
Business Pro,Low Value,135,3472.79
Business Pro,Medium Value,143,14436.35
Family Pack,High Value,153,38314.02
Family Pack,Low Value,154,3918.7
Family Pack,Medium Value,160,16094.74
Postpaid 30,High Value,138,36539.08
Postpaid 30,Low Value,131,3583.23
Postpaid 30,Medium Value,141,13587.48
Postpaid 50,High Value,119,29616.42


In [0]:
from pyspark.sql import Row

tables = [
    ("Bronze", "bronze_customers", "workspace.bronze.bronze_customers"),
    ("Bronze", "bronze_transactions", "workspace.bronze.bronze_transactions"),
    ("Silver", "silver_customers", "workspace.silver.silver_customers"),
    ("Silver", "silver_transactions", "workspace.silver.silver_transactions"),
    ("Gold", "gold_customer_value_segments", "workspace.gold.gold_customer_value_segments"),
    ("Gold", "gold_monthly_segment_performance", "workspace.gold.gold_monthly_segment_performance")
]

summary = []
for layer, table_name, full_name in tables:
    summary.append(Row(layer=layer, table_name=table_name, row_count=spark.table(full_name).count()))

pipeline_row_count_summary = spark.createDataFrame(summary)

display(pipeline_row_count_summary.orderBy("layer", "table_name"))

layer,table_name,row_count
Bronze,bronze_customers,2550
Bronze,bronze_transactions,12080
Gold,gold_customer_value_segments,2500
Gold,gold_monthly_segment_performance,5240
Silver,silver_customers,2500
Silver,silver_transactions,12000


In [0]:
from pyspark.sql import Row

lineage_rows = [
    Row(source_layer="Bronze", source_table="bronze_customers", target_layer="Silver", target_table="silver_customers", dependency="Silver customer table is created by cleaning and deduplicating Bronze customer records."),
    Row(source_layer="Bronze", source_table="bronze_transactions", target_layer="Silver", target_table="silver_transactions", dependency="Silver transaction table is created by standardising and validating Bronze transaction records."),
    Row(source_layer="Silver", source_table="silver_customers + silver_transactions", target_layer="Gold", target_table="gold_customer_value_segments", dependency="Gold customer value table aggregates transaction activity and joins customer attributes."),
    Row(source_layer="Silver", source_table="silver_transactions + silver_customers", target_layer="Gold", target_table="gold_monthly_segment_performance", dependency="Gold monthly segment table aggregates transaction activity by month, channel, plan, and state.")
]

medallion_dependency_summary = spark.createDataFrame(lineage_rows)
display(medallion_dependency_summary)

source_layer,source_table,target_layer,target_table,dependency
Bronze,bronze_customers,Silver,silver_customers,Silver customer table is created by cleaning and deduplicating Bronze customer records.
Bronze,bronze_transactions,Silver,silver_transactions,Silver transaction table is created by standardising and validating Bronze transaction records.
Silver,silver_customers + silver_transactions,Gold,gold_customer_value_segments,Gold customer value table aggregates transaction activity and joins customer attributes.
Silver,silver_transactions + silver_customers,Gold,gold_monthly_segment_performance,"Gold monthly segment table aggregates transaction activity by month, channel, plan, and state."
